# 개별종목 조합D — RandomForest

`기본모델/02.RandomForest.ipynb`과 같은 `models.random_forest.build_random_forest_baseline`을 가져오고
조합D 피처를 주입합니다. 기본모델 코드는 `models/`에 한 번만 존재합니다.
후보·라벨·날짜 그룹 12폴드 실행은 모든 조합이 같은 공통 함수를 사용합니다.


In [1]:
# 1. 기본모델을 가져옵니다.
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
if not (project_root / "pyproject.toml").is_file():
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from models.random_forest import build_random_forest_baseline  # noqa: E402

MODEL_NAME = 'RandomForest'
MODEL_BUILDER = build_random_forest_baseline


In [2]:
# 2. 조합D의 피처 값만 지정합니다.
import json

COMBINATION = 'D'
FEATURE_COLUMNS = (
    'atr_ratio',
    'hv_20',
    'range_1',
    'range_20',
    'bb_bandwidth',
    'volume_z_20',
    'turnover_20',
    'log_amihud_20',
)

report_path = project_root / "reports" / "stock_feature_combinations.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
print(f"조합{COMBINATION} 피처:", FEATURE_COLUMNS)
combination_report = report["combinations"].get(COMBINATION)
if combination_report is None:
    print("아직 실측 결과가 없습니다. 아래 공통 실행 명령으로 조합을 평가하세요.")
else:
    panel = combination_report["panel"]
    print("학습 기간:", panel["first_date"], "~", panel["last_date"])
    print("학습 행·종목:", panel["model_rows"], panel["stocks"])
    folds = pd.DataFrame(combination_report["outer_fold_results"])
    model_folds = folds.loc[folds["model"].eq(MODEL_NAME)].reset_index(drop=True)
    fold_columns = [
        "fold", "selected_class_weight", "train_dates", "valid_start", "valid_end",
        "accuracy", "training_majority_baseline_accuracy",
        "accuracy_minus_training_majority_baseline", "macro_f1", "balanced_accuracy",
        "mcc", "pr_auc_macro_ovr", "down_recall", "core_harmonic_mean",
    ]
    display(model_folds.loc[:, fold_columns].round(4))
    metric_columns = [
        "accuracy", "training_majority_baseline_accuracy",
        "accuracy_minus_training_majority_baseline", "macro_f1", "balanced_accuracy",
        "mcc", "pr_auc_macro_ovr", "down_recall", "core_harmonic_mean",
    ]
    display(model_folds.loc[:, metric_columns].mean().to_frame("OOS 폴드 평균").round(4))

# 조합별 노트북이 중복 학습하지 않도록 실제 fit은 공통 실행기에서 한 번 수행합니다.
print("재실행 명령: python scripts/run_stock_model_experiment.py")


조합D 피처: ('atr_ratio', 'hv_20', 'range_1', 'range_20', 'bb_bandwidth', 'volume_z_20', 'turnover_20', 'log_amihud_20')
학습 기간: 20110127 ~ 20240822
학습 행·종목: 159900 157


,fold,selected_class_weight,train_dates,valid_start,valid_end,accuracy,training_majority_baseline_accuracy,accuracy_minus_training_majority_baseline,macro_f1,balanced_accuracy,mcc,pr_auc_macro_ovr,down_recall,core_harmonic_mean
0,1,balanced,750,20140217,20140514,0.4167,0.5012,-0.0845,0.3396,0.3471,0.0284,0.3554,0.2621,0.3275
1,2,balanced,980,20150123,20150421,0.3684,0.3978,-0.0294,0.3496,0.3525,0.0357,0.3589,0.2682,0.3225
2,3,balanced,1210,20151228,20160328,0.3464,0.3762,-0.0298,0.3421,0.3423,0.0147,0.3423,0.3168,0.3346
3,4,balanced,1439,20161202,20170228,0.4129,0.4617,-0.0488,0.3552,0.3613,0.0532,0.3769,0.2455,0.3223
4,5,balanced,1669,20171113,20180207,0.3824,0.3901,-0.0077,0.3650,0.3682,0.0565,0.3830,0.3383,0.3610
5,6,balanced,1899,20181024,20190118,0.3718,0.3725,-0.0007,0.3695,0.3693,0.0573,0.3777,0.3358,0.3583
6,7,balanced,2129,20190930,20191224,0.4259,0.4781,-0.0523,0.3581,0.3634,0.0597,0.3731,0.2182,0.3085
7,8,balanced,2359,20200902,20201130,0.3429,0.3476,-0.0048,0.3407,0.3413,0.0162,0.3633,0.3354,0.3396
8,9,balanced,2589,20210806,20211105,0.3715,0.3914,-0.0199,0.3627,0.3672,0.0478,0.3735,0.3421,0.3584
9,10,balanced,2818,20220714,20221012,0.3596,0.3454,0.0141,0.3591,0.3622,0.0428,0.3600,0.3217,0.3459


,OOS 폴드 평균
accuracy,0.3800
training_majority_baseline_accuracy,0.3969
accuracy_minus_training_majority_baseline,-0.0168
macro_f1,0.3581
balanced_accuracy,0.3608
mcc,0.0459
pr_auc_macro_ovr,0.3678
down_recall,0.3102
core_harmonic_mean,0.3442


재실행 명령: python scripts/run_stock_model_experiment.py
